# MLP Specialization Analysis

Analyze how MLP layers specialize: input selectivity, output targeting,
cross-layer division of labor, sparsity profiles, and summary.

## Why This Matters

MLP specialization provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_specialization_analysis import (
    mlp_input_selectivity, mlp_output_targeting,
    mlp_cross_layer_division, mlp_activation_sparsity_profile,
    mlp_specialization_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Input Selectivity

How selective are MLP neurons to specific input patterns?

In [ ]:
for layer in range(4):
    result = mlp_input_selectivity(model, tokens, layer=layer, top_k=5)
    flag = ' [SELECTIVE]' if result['is_selective'] else ''
    print(f"  Layer {layer}: mean_sel={result['mean_selectivity']:.4f}{flag}")
    for n in result['per_neuron'][:3]:
        print(f"    N{n['neuron']}: sel={n['selectivity']:.3f}, "
              f"max_act={n['max_activation']:.4f}, top_pos={n['top_position']}")

## Output Targeting

Which vocabulary tokens does each neuron promote or suppress?

In [ ]:
for layer in range(4):
    result = mlp_output_targeting(model, tokens, layer=layer, top_k=5)
    print(f"  Layer {layer}: mean_range={result['mean_logit_range']:.4f}")
    for n in result['per_neuron'][:3]:
        print(f"    N{n['neuron']}: promotes tok{n['top_promoted_token']} "
              f"({n['top_promoted_logit']:+.3f}), suppresses tok{n['top_suppressed_token']} "
              f"({n['top_suppressed_logit']:+.3f})")

## Cross-Layer Division

Are MLP layers doing similar or different things?

In [ ]:
result = mlp_cross_layer_division(model, tokens)
print(f"Specialized: {result['is_specialized']}")
print(f"Mean cross-layer sim: {result['mean_cross_layer_similarity']:.4f}\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: sparsity={p['sparsity']:.4f}, norm={p['output_norm']:.4f}")
print()
for s in result['cross_layer_similarities']:
    print(f"  L{s['layer_i']}<->L{s['layer_j']}: cos={s['cosine']:.4f}")

## Activation Sparsity Profile

How sparse are MLP activations across layers and positions?

In [ ]:
result = mlp_activation_sparsity_profile(model, tokens)
print(f"Overall sparsity: {result['overall_sparsity']:.4f}")
print(f"Trend: {result['sparsity_trend']}\n")
for p in result['per_layer']:
    bar = '█' * int(p['mean_sparsity'] * 30)
    print(f"  Layer {p['layer']}: sparsity={p['mean_sparsity']:.4f}, "
          f"overlap={p['active_neuron_overlap']:.4f} {bar}")

## Specialization Summary

Cross-layer overview of MLP specialization.

In [ ]:
result = mlp_specialization_summary(model, tokens)
print(f"Most impactful layer: {result['most_impactful_layer']}\n")
for p in result['per_layer']:
    bar = '█' * int(min(p['logit_impact'] * 5, 30))
    print(f"  Layer {p['layer']}: norm={p['output_norm']:.4f}, "
          f"sparsity={p['sparsity']:.4f}, logit_impact={p['logit_impact']:.4f} {bar}")